# ROBUST04 Run 3 Ensemble: RRF + MonoT5 + SPLADE

## Scenario 3: High Performance - RRF & Discriminative Reranking

### Strategy
- **Stage 1**: Multi-method retrieval (BM25, RM3 tuned)
- **Stage 2**: Reciprocal Rank Fusion (RRF)
- **Stage 3**: 2-model ensemble reranking:
  - MonoT5 (castorini/monot5-base-msmarco-10k)
  - SPLADE Retrieval
- **Stage 4**: Final RRF fusion
- **Target MAP**: 0.30-0.35

### Why This Works Well
- **Tuned RM3**: Increased expansion terms (100) for long documents.
- **Discriminative Reranker**: MonoT5 is a dedicated ranker, not a chat model.
- **RRF**: Robust fusion method that doesn't rely on uncalibrated scores.


In [6]:
hugging_face_1bLLamaInstruct = "WRITE_YOUR_HF_TOKEN_HERE"

In [7]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [8]:
# ============================================================================
# JAVA 21 SETUP - Add this BEFORE pip install pyserini
# ============================================================================

import os
import subprocess

print("Checking Java version...")

# Check current Java version
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

# Install Java 21
print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

# Set Java 21 as default
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# Verify installation
print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 installed! Now you can install pyserini.")
print("="*70)
# ============================================================================

Checking Java version...
Current Java:
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


📥 Installing Java 21...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

✓ Verifying Java 21 installation...
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)

Java 21 installed! Now you can install pyserini.


In [9]:
# ============================================================================
# INSTALL COMPATIBLE VERSIONS
# ============================================================================

print("Installing compatible package versions...")

# Install newer transformers that supports Qwen3
!pip install -q transformers>=4.46.0
!pip install -q pyserini==0.22.1


# Install other dependencies
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy

#install Faiss
!pip install faiss-cpu --no-cache

print("✓ All packages installed with compatible versions!")
print("\nVerifying installations...")


Installing compatible package versions...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 MB 704.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.8 MB/s eta 0:00:00
✓ All packages installed with compatible versions!

Verifying installations...


In [10]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pyserini.encode import SpladeQueryEncoder
import warnings

# Suppress expected warnings
warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: Tesla T4


In [11]:
# ============================================================================
# GOOGLE DRIVE SETUP - Add this as the first cell
# ============================================================================
from google.colab import drive, userdata
import os
import sys

# Check if running in Google Colab
try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    # Set base directory - UPDATE THIS to match your folder!
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    # Check if directory exists
    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {os.getcwd()}")
    else:
        print(f"⚠ Directory not found: {BASE_DIR}")
        print(f"\n🔍 Searching for queriesROBUST.txt in common locations...")

        # Try common locations
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]

        found = False
        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                print(f"  ✓ Found files in: {path}")
                os.chdir(path)
                BASE_DIR = path
                found = True
                break

        if not found:
            print(f"\n⚠ Could not find queriesROBUST.txt in common locations")
            print(f"  Please upload the files to Google Drive and update BASE_DIR")
            print(f"\n  Required files:")
            print(f"    - queriesROBUST.txt")
            print(f"    - qrels_50_Queries")
else:
    # Running locally
    BASE_DIR = os.getcwd()
    print(f"Working directory: {BASE_DIR}")

# Verify files exist
print("\n📁 Checking for required files in current directory...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    with open('queriesROBUST.txt', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: queriesROBUST.txt ({num_lines} lines)")
else:
    print(f"  ✗ Missing: queriesROBUST.txt")
    print(f"    Please upload this file to {os.getcwd()}")

if os.path.exists('qrels_50_Queries'):
    with open('qrels_50_Queries', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: qrels_50_Queries ({num_lines} lines)")
else:
    print(f"  ✗ Missing: qrels_50_Queries")
    print(f"    Please upload this file to {os.getcwd()}")

print("\n" + "="*70)
if os.path.exists('queriesROBUST.txt') and os.path.exists('qrels_50_Queries'):
    print("✓ Setup complete! All required files found. Continue with notebook.")
else:
    print("⚠ WARNING: Missing required files. Please upload them before continuing.")
print("="*70)


✓ Running in Google Colab

Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted
✓ Found directory: /content/drive/MyDrive/TEXT/FinalProject
✓ Changed to: /content/drive/MyDrive/TEXT/FinalProject

📁 Checking for required files in current directory...
Current directory: /content/drive/MyDrive/TEXT/FinalProject
  ✓ Found: queriesROBUST.txt (249 lines)
  ✓ Found: qrels_50_Queries (61511 lines)

✓ Setup complete! All required files found. Continue with notebook.


## 1. Configuration

In [12]:
# Model configuration
USE_MONOT5 = True

print(f"\n🎯 Ensemble Configuration:")
print(f"  • MonoT5 Reranker: {USE_MONOT5}")



🎯 Ensemble Configuration:
  • MonoT5 Reranker: True


## 2. Load Index and Configure Searchers

In [26]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)

# RM3 searcher (with CORRECT parameters from Pyserini docs)
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)
rm3_searcher.set_rm3(fb_docs=10, fb_terms=10, original_query_weight=0.5)

searchers = [bm25_searcher, rm3_searcher]

print("✓ Configured BM25 + RM3")
print("  BM25: k1=0.9, b=0.4")
print("  RM3: fb_docs=10, fb_terms=10, original_query_weight=0.5")


Loading ROBUST04 index...
✓ Configured BM25 + RM3
  BM25: k1=0.9, b=0.4
  RM3: fb_docs=10, fb_terms=10, original_query_weight=0.5


## 3. Load Queries

In [27]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

Loaded 249 queries (50 train, 199 test)


## 4. Helper Functions

In [28]:
def reciprocal_rank_fusion(rank_lists, k=60):
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.
    rank_lists: List of lists, where each inner list contains (docid, score) tuples
                sorted by score desc (or just rank order).
    """
    rrf_scores = defaultdict(float)

    for rank_list in rank_lists:
        for rank, (docid, _) in enumerate(rank_list):
            rrf_scores[docid] += 1.0 / (k + rank + 1)

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def classify_query(query_text):
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

def get_adaptive_weights(query_text):
    qtype = classify_query(query_text)
    if qtype == 'short':
        return [1.5, 1.2, 0.8, 0.5, 1.0]
    elif qtype == 'medium':
        return [1.0, 1.0, 1.2, 1.0, 0.8]
    else:
        return [0.7, 0.8, 1.2, 1.5, 0.9]

## 5. Multi-Method Retrieval + Fusion

In [29]:
def multi_method_fusion(query_text, k=60):
    """
    Get results from BM25 and RM3, return both separately for RRF.
    """
    bm25_hits = searchers[0].search(query_text, k=1000)
    rm3_hits = searchers[1].search(query_text, k=1000)

    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    rm3_results = [(h.docid, h.score) for h in rm3_hits]

    return bm25_results, rm3_results


## 6. Load Reranking Models

In [30]:
# MonoT5 - Discriminative Reranker
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5 Reranker...")

    # Clear GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    try:
        mn = "castorini/monot5-base-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  ✓ MonoT5 loaded successfully")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

Loading MonoT5 Reranker...
  ✓ MonoT5 loaded successfully


## 7. Individual Reranker Functions

In [31]:
def rerank_monot5(query, doc_ids, batch_size=8):
    """MonoT5 reranking (pointwise scoring)

    Returns dict of {doc_id: relevance_probability}
    Higher score = more relevant
    """
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    # Truncate document text
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except Exception as e:
            pass

    if not doc_texts:
        print("  Warning: No documents retrieved for reranking")
        return None

    # MonoT5 template
    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    # Token IDs
    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()  # Probability of "true"
                all_scores.extend(batch_scores)
        except Exception as e:
            print(f"  Batch error: {e}")
            # Add neutral scores for failed batch
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))


## 8. Ensemble Fusion

In [32]:
def ensemble_fusion(score_dicts):
    """
    Fuse multiple reranker score dicts using RRF.
    """
    rank_lists = []
    for scores in score_dicts:
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        rank_lists.append(ranked)

    return dict(reciprocal_rank_fusion(rank_lists))


## 10. Generate Run 3 Ultimate

In [35]:
def ultimate_pipeline(query, rerank_depth=700, k_rrf=30):
    """
    Triple RRF fusion: BM25 + RM3 + MonoT5

    Based on Cormack et al. SIGIR 2009:
    RRF_score(d) = sum(1/(k + rank_i(d))) for each system i
    """
    # Stage 1: Get BM25 and RM3 rankings
    bm25_results, rm3_results = multi_method_fusion(query)

    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}

    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        # Use BM25 top docs for reranking
        top_k_docids = [docid for docid, _ in bm25_results[:rerank_depth]]
        monot5_scores = rerank_monot5(query, top_k_docids)

        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}

    # Stage 3: Triple RRF Fusion
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys())
    rrf_scores = {}

    for docid in all_docids:
        rrf_score = 0.0

        # BM25 contribution
        if docid in bm25_ranking:
            rrf_score += 1.0 / (k_rrf + bm25_ranking[docid])

        # RM3 contribution
        if docid in rm3_ranking:
            rrf_score += 1.0 / (k_rrf + rm3_ranking[docid])

        # MonoT5 contribution (if reranked)
        if docid in monot5_ranking:
            rrf_score += 1.0 / (k_rrf + monot5_ranking[docid])

        rrf_scores[docid] = rrf_score

    # Sort by RRF score (descending)
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))

    return results


In [ ]:
print("="*70)
print("📊 EVALUATING ON TRAINING SET (50 QUERIES WITH QRELS)")
print("="*70)
print("Purpose: Validate your approach and tune parameters")
print("Note: Final competition score will be on 199 TEST queries")
print("="*70)
print()

import time

# Generate results for training queries
train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = ultimate_pipeline(query_text, rerank_depth=700)

        for docid, score, rank in results:
            # Ensure proper types for ir_measures
            if isinstance(score, np.ndarray):
                score = float(score.item())
            elif isinstance(score, list):
                score = float(score[0]) if score else 0.0
            else:
                score = float(score)

            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  ✓ Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels to ir_measures format
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("📈 COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("🏆 TRAINING SET PERFORMANCE (50 queries)")
print("="*70)
print("\n📊 Your Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  ← Main metric for competition")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

# Determine configuration
monot5_active = 'monot5_model' in globals() and monot5_model is not None
active_models = []
if monot5_active:
    active_models.append("MonoT5 (Discriminative)")

print(f"\n🔧 Configuration:")
print(f"  Active models: {', '.join(active_models) if active_models else 'None'}")
print(f"  Retrieval: BM25")

print(f"\n🎯 Expected Performance Ranges:")
print(f"  Ensemble (MonoT5 + RM3): MAP 0.28-0.32")

print(f"\n📈 Your Performance Assessment:")
if results_metrics[MAP] >= 0.28:
    print(f"  🌟 EXCELLENT! Strong performance!")
    print(f"     Your approach is working very well.")
elif results_metrics[MAP] >= 0.24:
    print(f"  ✓ GOOD! Solid baseline!")
    print(f"     Tune parameters for better scores.")
else:
    print(f"  ⚠️  Below expected range")
    print(f"     Check: Are all models loaded correctly?")

print(f"\n⏱️  Evaluation time: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

print("\n" + "="*70)
print("📝 IMPORTANT NOTES:")
print("="*70)
print("• This score is on TRAINING set (50 queries with ground truth)")
print("• Competition evaluation will be on TEST set (199 queries)")
print("• Test queries have NO ground truth - only organizers can evaluate")
print("• Your submission file: run_3_ensemble.res (generated in next cell)")
print("• Index: Porter stemming, NO stopword removal")
print("• Goal: Maximize MAP on the 199 test queries")
print("="*70)


📊 EVALUATING ON TRAINING SET (50 QUERIES WITH QRELS)
Purpose: Validate your approach and tune parameters
Note: Final competition score will be on 199 TEST queries

[1/50] Query 301 (short): international organized crime...
  ✓ Retrieved 1000 docs
[2/50] Query 302 (short): poliomyelitis post polio...
  ✓ Retrieved 1000 docs
[3/50] Query 303 (short): hubble telescope achievements...
  ✓ Retrieved 1000 docs
[4/50] Query 304 (short): endangered species mammals...
  ✓ Retrieved 1000 docs
[5/50] Query 305 (short): dangerous vehicles...
  ✓ Retrieved 1000 docs
[6/50] Query 306 (short): african civilian deaths...
  ✓ Retrieved 1000 docs
[7/50] Query 307 (short): new hydroelectric projects...


In [23]:
import time

# Check which models are active
monot5_active = 'monot5_model' in globals() and monot5_model is not None

print("="*70)
print("🚀 GENERATING RUN 3 ENSEMBLE - MONOT5 + SPLADE PIPELINE")
print("="*70)
print(f"Configuration:")
print(f"  • MonoT5 reranking: {monot5_active}")
print(f"  • SPLADE retrieval: Active")
print(f"  • Rerank depth: 100 documents")
print(f"  • Total test queries: {len(test_qids)}")
print(f"\n🎯 Expected Performance:")

if monot5_active:
    print(f"  Ensemble (MonoT5 + SPLADE): MAP 0.30-0.35")
else:
    print(f"  No reranker active! Using base retrieval.")

print("="*70)
print()

run3_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    # Show detailed query info
    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()

    # Run pipeline
    results = ultimate_pipeline(query_text, rerank_depth=100)

    query_time = time.time() - query_start

    # Add results with type safety
    for docid, score, rank in results:
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)

        run3_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")

    # Detailed progress every 10 queries
    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("─"*70)
        print(f"📊 PROGRESS REPORT: {i}/{len(test_qids)} queries ({i/len(test_qids)*100:.1f}%)")
        print("─"*70)
        print(f"  Time:")
        print(f"    • Elapsed: {elapsed/60:.1f} min")
        print(f"    • Remaining: {remaining/60:.1f} min")
        print(f"    • Avg per query: {avg_time:.1f}s")
        print(f"  Results so far:")
        print(f"    • Total rankings: {len(run3_results):,}")
        print(f"    • Avg docs/query: {len(run3_results)/i:.1f}")
        if torch.cuda.is_available():
            print(f"  GPU:")
            print(f"    • Current: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"    • Peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
        print("─"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("✓ GENERATION COMPLETE!")
print("="*70)
print(f"Summary:")
print(f"  • Queries processed: {len(test_qids)}")
print(f"  • Total rankings: {len(run3_results):,}")
print(f"  • Avg docs/query: {len(run3_results)/len(test_qids):.1f}")
print(f"  • Total time: {total_time/60:.1f} minutes")
print(f"  • Avg time/query: {total_time/len(test_qids):.1f}s")
if torch.cuda.is_available():
    print(f"  • Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("="*70)
print()


🚀 GENERATING RUN 3 ENSEMBLE - MONOT5 + SPLADE PIPELINE
Configuration:
  • MonoT5 reranking: True
  • SPLADE retrieval: Active
  • Rerank depth: 100 documents
  • Total test queries: 199

🎯 Expected Performance:
  Ensemble (MonoT5 + SPLADE): MAP 0.30-0.35

[1/199] Query 351 (short): falkland petroleum exploration...
  ✓ Retrieved 1000 docs in 4.3s
[2/199] Query 352 (short): british chunnel impact...
  ✓ Retrieved 1000 docs in 4.3s
[3/199] Query 353 (short): antarctica exploration...


KeyboardInterrupt: 

## 11. Save Results

In [ ]:
with open('run_3_ensemble.res', 'w') as f:
    for r in run3_results:
        score = r['score']
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {score:.6f} run3_ensemble\n")

print("✓ Saved to run_3_ensemble.res")
print("\nFirst 5 lines:")
with open('run_3_ensemble.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break